# Feature Engineering

## Objective

Transform the cleaned restaurant dataset into structured features that can be used by the recommendation engine.

Outputs:

- Cuisine
- Restaurant Type
- Experience
- Expanded Attributes

In [1]:
import pandas as pd

restaurants = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/processed/restaurants_clean.csv")

taxonomy = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/reference/category_mapping.csv", encoding="cp1252")

In [2]:
restaurants.head(3)

,business_id,name,city,state,postal_code,latitude,longitude,stars,review_count,categories,attributes,hours,is_open
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'RestaurantsDelivery': 'False', 'OutdoorSeati...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ...",1
1,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...","{'BusinessParking': 'None', 'BusinessAcceptsCr...","{'Monday': '0:0-0:0', 'Tuesday': '6:0-22:0', '...",1
2,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,Affton,MO,63123,38.565165,-90.321087,3.0,19,"Pubs, Restaurants, Italian, Bars, American (Tr...","{'Caters': 'True', 'Alcohol': ""u'full_bar'"", '...",Uknown,0


In [3]:
taxonomy.head(3)

,Category,Count,Feature Type,Standardized Value,Keep,Notes
0,Restaurants,52268,Generic,NaN,No,NaN
1,Food,15472,Generic,NaN,No,NaN
2,Nightlife,8723,Experience,NaN,Yes,NaN


In [4]:
taxonomy_keep = taxonomy[taxonomy["Keep"] == "Yes"]

In [5]:
mapping = taxonomy_keep.set_index("Category").to_dict("index")

Test

In [6]:
restaurants.iloc[0]["categories"]

'Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries'

In [7]:
categories = [
    c.strip()
    for c in restaurants.iloc[0]["categories"].split(",")
]

In [8]:
for c in categories:
    if c in mapping:
        print(c, "->", mapping[c])

Bubble Tea -> {'Count': 277, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Bubble Tea', 'Keep': 'Yes', 'Notes': nan}
Coffee & Tea -> {'Count': 4053, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Cafe', 'Keep': 'Yes', 'Notes': nan}
Bakeries -> {'Count': 1889, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Bakery', 'Keep': 'Yes', 'Notes': nan}


In [9]:
def extract_features(category_string, mapping):

    features = {
        "Cuisine": set(),
        "Restaurant Type": set(),
        "Experience": set()
    }

    categories = [
        category.strip()
        for category in category_string.split(",")
    ]

    for category in categories:

        if category in mapping:

            info = mapping[category]

            feature_type = info["Feature Type"]

            value = info["Standardized Value"]

            if feature_type in features and pd.notna(value):
                features[feature_type].add(value)

    return {
        key: list(value)
        for key, value in features.items()
    }

In [10]:
restaurant = restaurants.iloc[0]["categories"]

extract_features(restaurant,mapping)



{'Cuisine': [],
 'Restaurant Type': ['Bakery', 'Cafe', 'Bubble Tea'],
 'Experience': []}

In [11]:
restaurants["Extracted Features"] = restaurants["categories"].apply(
    lambda x: extract_features(x, mapping)
)

In [12]:
restaurants[["name", "Extracted Features"]].head()

,name,Extracted Features
0,St Honore Pastries,"{'Cuisine': [], 'Restaurant Type': ['Bakery', ..."
1,Sonic Drive-In,"{'Cuisine': [], 'Restaurant Type': ['Burgers',..."
2,Tsevi's Pub And Grill,"{'Cuisine': ['Italian', 'Greek', 'American'], ..."
3,Sonic Drive-In,"{'Cuisine': [], 'Restaurant Type': ['Ice Cream..."
4,Vietnamese Food Truck,"{'Cuisine': ['Vietnamese'], 'Restaurant Type':..."


In [13]:
restaurants["Extracted Features"].iloc[0]

{'Cuisine': [],
 'Restaurant Type': ['Bakery', 'Cafe', 'Bubble Tea'],
 'Experience': []}

In [14]:
restaurants["categories"].iloc[0]

'Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeries'

In [15]:
restaurants["Restaurant Type"] = restaurants["Extracted Features"].apply(
    lambda features: features["Restaurant Type"]
)

restaurants["Experience"] = restaurants["Extracted Features"].apply(
    lambda features: features["Experience"]
)

restaurants["Cuisine"] = restaurants["Extracted Features"].apply(
    lambda features: features["Cuisine"]
)

In [16]:
restaurants[
    ["name", "Cuisine", "Restaurant Type", "Experience"]
].head()

,name,Cuisine,Restaurant Type,Experience
0,St Honore Pastries,[],"[Bakery, Cafe, Bubble Tea]",[]
1,Sonic Drive-In,[],"[Burgers, Sandwiches, Ice Cream & Frozen Yogur...",[]
2,Tsevi's Pub And Grill,"[Italian, Greek, American]",[],[]
3,Sonic Drive-In,[],"[Ice Cream & Frozen Yogurt, Burgers, Fast Food]",[]
4,Vietnamese Food Truck,[Vietnamese],[],[]


In [17]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

cuisine_mlb = mlb.fit_transform(restaurants["Cuisine"])

cuisine_mlb

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(52268, 98))

In [18]:
Cuisine = mlb.inverse_transform(cuisine_mlb)

Cuisine

[(),
 (),
 ('American', 'Greek', 'Italian'),
 (),
 ('Vietnamese',),
 ('American',),
 ('Italian',),
 ('Japanese',),
 ('Korean',),
 (),
 ('Asian Fusion',),
 ('Italian',),
 (),
 ('Japanese',),
 ('Italian',),
 (),
 (),
 (),
 ('American',),
 (),
 ('Italian',),
 ('Chinese',),
 ('American',),
 ('Italian',),
 (),
 ('American', 'Italian'),
 (),
 (),
 ('American',),
 (),
 ('American',),
 ('American',),
 ('American',),
 ('Italian',),
 ('American',),
 (),
 ('Asian Fusion', 'Japanese'),
 (),
 (),
 (),
 (),
 ('Italian',),
 (),
 ('American', 'Italian'),
 (),
 ('Cajun/Creole',),
 ('Mexican',),
 ('French', 'Mediterranean', 'Moroccan'),
 ('American',),
 (),
 (),
 ('American',),
 ('Chinese',),
 ('Italian',),
 (),
 (),
 ('Filipino',),
 ('Japanese',),
 ('Mexican',),
 ('Mexican',),
 (),
 (),
 ('Italian',),
 ('American',),
 (),
 ('Japanese',),
 (),
 (),
 (),
 (),
 ('American',),
 (),
 ('Chinese', 'Japanese', 'Thai'),
 (),
 (),
 ('Southern',),
 ('American',),
 ('Japanese',),
 ('Hawaiian',),
 ('Irish',),
 (),


In [19]:
type(cuisine_mlb)

numpy.ndarray

In [20]:
mlb.classes_

array(['Afghan', 'African', 'American', 'Arabic', 'Argentine', 'Armenian',
       'Asian Fusion', 'Australian', 'Austrian', 'Bangladeshi', 'Basque',
       'Belgian', 'Brazilian', 'British', 'Burmese', 'Cajun/Creole',
       'Calabrian', 'Cambodian', 'Canadian (New)', 'Cantonese',
       'Caribbean', 'Chinese', 'Colombian', 'Cuban', 'Czech', 'Dominican',
       'Eastern European', 'Egyptian', 'Ethiopian', 'Filipino', 'French',
       'Georgian', 'German', 'Greek', 'Guamanian', 'Hainan', 'Haitian',
       'Hawaiian', 'Himalayan/Nepalese', 'Honduran', 'Hungarian',
       'Iberian', 'Indian', 'Indonesian', 'Irish', 'Israeli', 'Italian',
       'Izakaya', 'Japanese', 'Korean', 'Kosher', 'Laotian',
       'Latin American', 'Lebanese', 'Malaysian', 'Mediterranean',
       'Mexican', 'Middle Eastern', 'Modern European', 'Mongolian',
       'Moroccan', 'New Mexican', 'Nicaraguan', 'Oriental', 'Pakistani',
       'Pan Asian', 'Persian/Iranian', 'Peruvian', 'Polish', 'Portuguese',
       'Puerto

In [21]:
cuisine_df = pd.DataFrame(cuisine_mlb,columns=mlb.classes_)

In [22]:
cuisine_df

,Afghan,African,American,Arabic,Argentine,Armenian,Asian Fusion,Australian,Austrian,Bangladeshi,...,Taiwanese,Tex-Mex,Thai,Trinidadian,Turkish,Tuscan,Ukrainian,Uzbek,Venezuelan,Vietnamese
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52263,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52264,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52266,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Restaurant Type Matrix

In [23]:
restaurants["Restaurant Type"].apply(type).value_counts()

Restaurant Type
<class 'list'>    52268
Name: count, dtype: int64

In [24]:
all_types = set()

for lst in restaurants["Restaurant Type"]:
    all_types.update(type(x) for x in lst)

all_types

{str}

In [25]:
restaurants[
    restaurants["Restaurant Type"].apply(
        lambda lst: any(not isinstance(x, str) for x in lst)
    )
][["name", "Restaurant Type"]]

,name,Restaurant Type


In [26]:
restaurant_type_mlb = mlb.fit_transform(restaurants["Restaurant Type"])

restaurant_type = mlb.inverse_transform(restaurant_type_mlb)

restaurant_type_df = pd.DataFrame(restaurant_type_mlb, columns=mlb.classes_)

restaurant_type_df

,Acai Bowls,Alcoholic Beverages,Bagels,Bakery,Barbeque,Beer,Breakfast & Brunch,Bubble Tea,Buffets,Burgers,...,Tapas/Small Plates,Tea,Teppanyaki,Tonkatsu,Vegan,Vegetarian,Waffles,Whiskey,Wine & Spirits,Wraps
0,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52263,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52264,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52266,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Experience Matrix

In [27]:
experience_mlb = mlb.fit_transform(restaurants["Experience"])

experience = mlb.inverse_transform(experience_mlb)

experience_df = pd.DataFrame(experience_mlb, columns=mlb.classes_)

experience_df

,Beer,Lounges,Wine
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
...,...,...,...
52263,0,0,0
52264,0,0,0
52265,0,0,0
52266,0,0,0


Numeric DataFrame

In [28]:
restaurants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52268 entries, 0 to 52267
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   business_id         52268 non-null  object 
 1   name                52268 non-null  object 
 2   city                52268 non-null  object 
 3   state               52268 non-null  object 
 4   postal_code         52247 non-null  object 
 5   latitude            52268 non-null  float64
 6   longitude           52268 non-null  float64
 7   stars               52268 non-null  float64
 8   review_count        52268 non-null  int64  
 9   categories          52268 non-null  object 
 10  attributes          52268 non-null  object 
 11  hours               52268 non-null  object 
 12  is_open             52268 non-null  int64  
 13  Extracted Features  52268 non-null  object 
 14  Restaurant Type     52268 non-null  object 
 15  Experience          52268 non-null  object 
 16  Cuis

In [29]:
numeric_features_df = restaurants[
    [
        "latitude",
        "longitude",
        "stars",
        "review_count"
    ]
]

In [30]:
numeric_features_df

,latitude,longitude,stars,review_count
0,39.955505,-75.155564,4.0,80
1,36.269593,-87.058943,2.0,6
2,38.565165,-90.321087,3.0,19
3,36.208102,-86.768170,1.5,10
4,27.955269,-82.456320,4.0,10
...,...,...,...,...
52263,39.925656,-75.310344,3.0,11
52264,43.615401,-116.284689,4.0,33
52265,39.935982,-75.158665,4.5,35
52266,39.856185,-75.427725,4.5,14


In [31]:
master_feature_matrix = pd.concat([cuisine_df, restaurant_type_df, experience_df, numeric_features_df], axis=1)

master_feature_matrix

,Afghan,African,American,Arabic,Argentine,Armenian,Asian Fusion,Australian,Austrian,Bangladeshi,...,Whiskey,Wine & Spirits,Wraps,Beer,Lounges,Wine,latitude,longitude,stars,review_count
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,39.955505,-75.155564,4.0,80
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,36.269593,-87.058943,2.0,6
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,38.565165,-90.321087,3.0,19
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,36.208102,-86.768170,1.5,10
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,27.955269,-82.456320,4.0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52263,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,39.925656,-75.310344,3.0,11
52264,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,43.615401,-116.284689,4.0,33
52265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,39.935982,-75.158665,4.5,35
52266,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,39.856185,-75.427725,4.5,14


In [32]:
master_feature_matrix.shape

(52268, 183)

In [33]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(numeric_features_df[["stars","review_count"]])

scaled_numeric

scaled_numeric_df = pd.DataFrame(scaled_numeric, columns = ['stars', 'review_count'])

scaled_numeric_df


,stars,review_count
0,0.584422,-0.038463
1,-1.826420,-0.430126
2,-0.620999,-0.361321
3,-2.429131,-0.408955
4,0.584422,-0.408955
...,...,...
52263,-0.620999,-0.403662
52264,0.584422,-0.287222
52265,1.187133,-0.276637
52266,1.187133,-0.387784


In [34]:
recommendation_feature_matrix = pd.concat([cuisine_df, restaurant_type_df, experience_df, scaled_numeric_df], axis=1)

recommendation_feature_matrix

,Afghan,African,American,Arabic,Argentine,Armenian,Asian Fusion,Australian,Austrian,Bangladeshi,...,Vegetarian,Waffles,Whiskey,Wine & Spirits,Wraps,Beer,Lounges,Wine,stars,review_count
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.584422,-0.038463
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-1.826420,-0.430126
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-0.620999,-0.361321
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-2.429131,-0.408955
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.584422,-0.408955
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52263,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-0.620999,-0.403662
52264,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.584422,-0.287222
52265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1.187133,-0.276637
52266,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1.187133,-0.387784


In [35]:
recommendation_feature_matrix.shape

(52268, 181)

In [36]:
def find_restaurant(restaurant_name, city):

    condition = (restaurants["name"] == restaurant_name) & (restaurants["city"] == city)

    return(

    restaurants[condition].index

    )
    

saved_index = find_restaurant("Sonic Drive-In","Nashville")

selected_cols = ["name","city","stars","review_count"]

restaurants.loc[saved_index, selected_cols]

,name,city,stars,review_count
3,Sonic Drive-In,Nashville,1.5,10
1837,Sonic Drive-In,Nashville,3.0,20
3264,Sonic Drive-In,Nashville,2.5,28
10088,Sonic Drive-In,Nashville,1.5,24
15153,Sonic Drive-In,Nashville,2.0,32
21934,Sonic Drive-In,Nashville,2.0,13
24257,Sonic Drive-In,Nashville,1.5,11
27560,Sonic Drive-In,Nashville,3.0,16
31134,Sonic Drive-In,Nashville,1.5,40
38458,Sonic Drive-In,Nashville,2.0,12


In [37]:
recommendation_feature_matrix.index

RangeIndex(start=0, stop=52268, step=1)

In [38]:
recommendation_feature_matrix.head()

,Afghan,African,American,Arabic,Argentine,Armenian,Asian Fusion,Australian,Austrian,Bangladeshi,...,Vegetarian,Waffles,Whiskey,Wine & Spirits,Wraps,Beer,Lounges,Wine,stars,review_count
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.584422,-0.038463
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-1.826420,-0.430126
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-0.620999,-0.361321
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,-2.429131,-0.408955
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0.584422,-0.408955


In [39]:
vector = recommendation_feature_matrix.iloc[3264]

vector.shape

(181,)

In [40]:
vector = recommendation_feature_matrix.iloc[[3264]]

vector.shape

(1, 181)

In [41]:
def get_feature_vector(index):

    vector = recommendation_feature_matrix.iloc[[index]]


    return(

        vector, index


    )

selected_vector, selected_index = get_feature_vector(3264)

selected_vector.shape


(1, 181)

In [42]:
print(selected_index)

3264


In [43]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_scores = cosine_similarity(
    selected_vector,
    recommendation_feature_matrix
)

In [44]:
similarity_df = pd.DataFrame(similarity_scores.T, columns=["similarity"])

similarity_df = similarity_df.drop(selected_index)

similarity_df.head()

,similarity
0,-0.179375
1,0.913382
2,0.217233
3,0.945043
4,-0.222869


In [45]:
similarity_sorted = similarity_df.sort_values(by="similarity", ascending=False)

similarity_sorted

,similarity
6355,1.000000
6255,0.999973
30466,0.999973
22047,0.999952
3275,0.999952
...,...
32889,-0.587463
49143,-0.589004
25792,-0.589031
40400,-0.589262


In [46]:
restaurants.loc[[3264, 6355]]


,business_id,name,city,state,postal_code,latitude,longitude,stars,review_count,categories,attributes,hours,is_open,Extracted Features,Restaurant Type,Experience,Cuisine
3264,HlSve6o5TAXbl4UDwCfcFw,Sonic Drive-In,Nashville,TN,37221,36.078544,-86.953119,2.5,28,"Ice Cream & Frozen Yogurt, Food, Restaurants, ...","{'RestaurantsTakeOut': 'True', 'Ambience': ""{'...","{'Monday': '0:0-0:0', 'Tuesday': '7:0-22:0', '...",1,"{'Cuisine': [], 'Restaurant Type': ['Ice Cream...","[Ice Cream & Frozen Yogurt, Burgers, Fast Food]",[],[]
6355,nv7i6LwawjPxlmxaKJWuJg,Dairy Queen,Tucson,AZ,85730,32.192417,-110.826305,2.5,28,"Restaurants, Fast Food, Food, Ice Cream & Froz...","{'WiFi': ""'no'"", 'DriveThru': 'True', 'BikePar...","{'Monday': '10:0-22:0', 'Tuesday': '10:0-22:0'...",1,"{'Cuisine': [], 'Restaurant Type': ['Ice Cream...","[Ice Cream & Frozen Yogurt, Burgers, Fast Food]",[],[]
